# ADK 도구(Tools)와 통합

에이전트는 **도구**를 통해 검색, 캘린더, 메일 등과 연결됩니다.

- 통합 카탈로그: [Tools and Integrations](https://google.github.io/adk-docs/integrations/)
- 커스텀 도구: [Custom Tools](https://google.github.io/adk-docs/tools-custom/)

이 노트북은 **Gemini** + [Google AI Studio](https://aistudio.google.com/apikey) **무료 API 키**를 사용합니다.

1. **Google Search** (`google_search`) — Gemini와 함께 쓰는 내장 검색  
2. **Gmail / Calendar 툴셋** — OAuth·ADC 필요 (선택 데모)  
3. **FunctionTool** — 직접 만든 Python 함수

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# .env 안의 키를 os.environ 에 올려 둡니다 (노트북 cwd에 따라 경로 2곳 시도)
load_dotenv(Path(".env").resolve())
load_dotenv(Path("..") / ".env")

# Gemini API 호출에 필요 (AI Studio에서 발급)
assert os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY"), (
    ".env 에 GOOGLE_API_KEY 또는 GEMINI_API_KEY 를 설정하세요."
)

## 1) Google Search (`google_search`)

`google.adk.tools.google_search`는 **Gemini 2 계열**에서 모델이 Google 검색을 활용하도록 설정합니다. 로컬에서 검색 API를 직접 호출하는 것과는 다릅니다.

- **Gemini 모델**과 함께 사용해야 합니다.
- 검색은 API 할당량·정책의 영향을 받을 수 있습니다.

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types

APP = "search_tool_demo"
USER = "u1"
SESSION = "s1"

# tools=[...]: 모델이 "도구 호출" 형태로 쓸 수 있는 기능 목록
# google_search: 미리 만들어진 싱글톤 객체 (별도 인스턴스 생성 불필요)
search_agent = LlmAgent(
    model="gemini-2.0-flash",
    name="search_agent",
    instruction=(
        "사용자 질문에 답할 때 필요하면 google_search 도구로 최신 정보를 확인하세요. "
        "한국어로 답하세요."
    ),
    tools=[google_search],
)

session_service = InMemorySessionService()


async def run_turn(question: str) -> None:
    """한 질문만 처리하고 세션을 삭제합니다 (다음 실습과 섞이지 않게)."""
    await session_service.create_session(
        app_name=APP, user_id=USER, session_id=SESSION, state={}
    )
    runner = Runner(
        agent=search_agent, app_name=APP, session_service=session_service
    )
    msg = types.Content(role="user", parts=[types.Part(text=question)])
    async for event in runner.run_async(
        user_id=USER, session_id=SESSION, new_message=msg
    ):
        if event.is_final_response() and event.content and event.content.parts:
            t = event.content.parts[0].text
            if t:
                print(t)
    await session_service.delete_session(
        app_name=APP, user_id=USER, session_id=SESSION
    )


await run_turn("최근 한국에서 주목받는 AI 뉴스 한 가지를 한 줄로 요약해줘.")

## 2) Gmail · Google Calendar 툴셋

툴셋 생성 시 API 스펙 로딩 등으로 **Application Default Credentials**가 필요할 수 있습니다. 실제 호출에는 **OAuth 클라이언트** 설정이 추가로 필요합니다.

In [ ]:
import os

from google.adk.tools.google_api_tool import CalendarToolset, GmailToolset

# Google Cloud Console의 OAuth 2.0 클라이언트 ID/비밀 (웹 또는 데스크톱 앱)
cid = os.environ.get("GOOGLE_OAUTH_CLIENT_ID")
csecret = os.environ.get("GOOGLE_OAUTH_CLIENT_SECRET")


async def _show_tool_counts() -> None:
    """Discovery API 기반으로 수많은 메서드가 생기므로 tool_filter로 일부만 노출합니다."""
    gmail_tools = GmailToolset(
        client_id=cid,
        client_secret=csecret,
        tool_filter=["gmail.users.messages.list", "gmail.users.messages.get"],
    )
    calendar_tools = CalendarToolset(
        client_id=cid,
        client_secret=csecret,
        tool_filter=["calendar.events.list"],
    )
    # get_tools: 비동기 — OpenAPI 스펙에서 실제 호출 가능한 Tool 객체 리스트 생성
    g = await gmail_tools.get_tools()
    c = await calendar_tools.get_tools()
    print("Gmail tools:", len(g))
    print("Calendar tools:", len(c))


if not cid or not csecret:
    print(
        "Gmail/Calendar 데모 생략: GOOGLE_OAUTH_CLIENT_ID / "
        "GOOGLE_OAUTH_CLIENT_SECRET 설정 후 실행하세요."
    )
else:
    try:
        await _show_tool_counts()
    except Exception as e:
        # ADC 미설정·네트워크 등으로 여기서 실패할 수 있음
        print("툴셋 초기화 실패:", type(e).__name__, e)

## 3) 커스텀 도구: `FunctionTool`

함수 **이름·docstring**이 곧 모델에게 보이는 도구 설명이 됩니다.

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import FunctionTool
from google.genai import types


def square(n: int) -> dict:
    """정수 n의 제곱 n*n을 반환합니다.

    모델은 이 docstring을 읽고 "언제·어떤 인자로" 호출할지 추론합니다.

    Args:
        n: 정수.

    Returns:
        {'n': 입력값, 'square': 제곱값}
    """
    return {"n": n, "square": n * n}


# Python 함수를 ADK 도구로 감쌉니다.
square_tool = FunctionTool(func=square)

math_agent = LlmAgent(
    model="gemini-2.0-flash",
    name="math_tool_agent",
    instruction="제곱 계산이 필요하면 square 도구를 사용하세요. 한국어로 답하세요.",
    tools=[square_tool],
)

APP2 = "function_tool_demo"
svc = InMemorySessionService()


async def ask_math() -> None:
    await svc.create_session(
        app_name=APP2, user_id="u", session_id="s", state={}
    )
    r = Runner(agent=math_agent, app_name=APP2, session_service=svc)
    q = types.Content(role="user", parts=[types.Part(text="12의 제곱은?")])
    async for ev in r.run_async(user_id="u", session_id="s", new_message=q):
        if ev.is_final_response() and ev.content and ev.content.parts:
            tx = ev.content.parts[0].text
            if tx:
                print(tx)
    await svc.delete_session(app_name=APP2, user_id="u", session_id="s")


await ask_math()